# 7.2 — Custom Transformers

**Chapter 7, section 7.2.3.** The starting point for **Exercise 8 (b) and (c)**.

**The question this notebook answers:** what is the smallest thing that can be a pipeline stage,
and what does that smallest thing not do?

A transformer is an object with a `_transform` method that takes a DataFrame and returns one.
That is the entire contract, and it is short enough to write out. The notebook writes two of
them beside their built-in equivalents, drops one into a `Pipeline`, and then tries to save the
pipeline — which fails, for a reason that is the point of the second half.

Source: `Notebooks old/Spark-Example-25a-Mllib-Custom-Transformer.ipynb` (contributed by Tommy
Lin, October 2024), modernized and revised.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-7.2")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


---

## 1. `Tokenizer`, built in

Three sentences. The built-in `Tokenizer` lowercases the string and splits it on spaces, and it
**appends** a column rather than replacing one — which is what every transformer does.

In [2]:
from pyspark.ml.feature import Tokenizer

text = [("He went to the store.",),
        ("Dogs like to fight cats.",),
        ("When did you do that?",)]
df = spark.createDataFrame(text, ["text"])

builtin = Tokenizer(inputCol="text", outputCol="words")
builtin.transform(df).show(truncate=False)

+------------------------+------------------------------+
|text                    |words                         |
+------------------------+------------------------------+
|He went to the store.   |[he, went, to, the, store.]   |
|Dogs like to fight cats.|[dogs, like, to, fight, cats.]|
|When did you do that?   |[when, did, you, do, that?]   |
+------------------------+------------------------------+



## …and written by hand

Four lines of body. `_transform` receives the DataFrame and returns it with one more column.
Nothing else is required, and nothing else is happening.

In [3]:
from pyspark.ml import Transformer
import pyspark.sql.functions as F


class CustomTokenizer(Transformer):
    def __init__(self, inputCol, outputCol):
        super().__init__()
        self.inputCol = inputCol
        self.outputCol = outputCol

    # Takes the input column, performs the operation, appends the result.
    def _transform(self, df):
        return df.withColumn(self.outputCol,
                             F.split(F.lower(F.col(self.inputCol)), " "))


custom = CustomTokenizer(inputCol="text", outputCol="words")
custom.transform(df).show(truncate=False)

+------------------------+------------------------------+
|text                    |words                         |
+------------------------+------------------------------+
|He went to the store.   |[he, went, to, the, store.]   |
|Dogs like to fight cats.|[dogs, like, to, fight, cats.]|
|When did you do that?   |[when, did, you, do, that?]   |
+------------------------+------------------------------+



In [4]:
# The two agree. An assertion is worth more here than a sentence claiming equivalence,
# because it fails loudly if a later Spark release changes what Tokenizer does.
a = builtin.transform(df).select("words").collect()
b = custom.transform(df).select("words").collect()
assert a == b
print("built-in and hand-written tokenizer produce the same column")

built-in and hand-written tokenizer produce the same column


---

## 2. `VectorAssembler`, and where the hand-written version stops being equivalent

The same exercise with the assembler is more instructive because the hand-written version is
**not** equivalent, and the difference is worth seeing.

In [5]:
from pyspark.ml.feature import VectorAssembler

x = [(1.0, 2.4), (2.9, 5.0), (3.3, 1.0), (4.8, 2.0), (5.0, 4.6), (6.0, 1.2)]
dfx = spark.createDataFrame(x, ["x1", "x2"])

assembler = VectorAssembler(inputCols=["x1", "x2"], outputCol="features")
assembler.transform(dfx).show(truncate=False)

+---+---+---------+
|x1 |x2 |features |
+---+---+---------+
|1.0|2.4|[1.0,2.4]|
|2.9|5.0|[2.9,5.0]|
|3.3|1.0|[3.3,1.0]|
|4.8|2.0|[4.8,2.0]|
|5.0|4.6|[5.0,4.6]|
|6.0|1.2|[6.0,1.2]|
+---+---+---------+



In [6]:
class CustomVectorAssembler(Transformer):
    def __init__(self, inputCols, outputCol):
        super().__init__()
        self.inputCols = inputCols
        self.outputCol = outputCol

    def _transform(self, df):
        return df.withColumn(self.outputCol, F.array(*self.inputCols))


CustomVectorAssembler(inputCols=["x1", "x2"], outputCol="features").transform(dfx).show(
    truncate=False)

+---+---+----------+
|x1 |x2 |features  |
+---+---+----------+
|1.0|2.4|[1.0, 2.4]|
|2.9|5.0|[2.9, 5.0]|
|3.3|1.0|[3.3, 1.0]|
|4.8|2.0|[4.8, 2.0]|
|5.0|4.6|[5.0, 4.6]|
|6.0|1.2|[6.0, 1.2]|
+---+---+----------+



In [7]:
# The values match; the types do not.
built = assembler.transform(dfx)
hand = CustomVectorAssembler(inputCols=["x1", "x2"], outputCol="features").transform(dfx)

print("built-in :", dict(built.dtypes)["features"])
print("by hand  :", dict(hand.dtypes)["features"])

built-in : vector
by hand  : array<double>


`F.array` produces `array<double>`. MLlib estimators require a `VectorUDT`, so the hand-written
assembler would be rejected by the very next stage of a real pipeline. The lesson is not that
the four-line form is wrong but that **the contract a stage has to satisfy includes the schema
it produces**, and a transformer that appends the wrong type raises nothing until something
downstream reads it.

---

## 3. A custom stage inside a pipeline

A custom transformer is a pipeline stage like any other. Nothing in `Pipeline` knows or cares
that this class was defined in a notebook cell thirty seconds ago.

In [8]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StopWordsRemover

pipeline = Pipeline(stages=[
    CustomTokenizer(inputCol="text", outputCol="words"),
    StopWordsRemover(inputCol="words", outputCol="filtered"),
])

model = pipeline.fit(df)
model.transform(df).select("text", "filtered").show(truncate=False)

+------------------------+--------------------------+
|text                    |filtered                  |
+------------------------+--------------------------+
|He went to the store.   |[went, store.]            |
|Dogs like to fight cats.|[dogs, like, fight, cats.]|
|When did you do that?   |[that?]                   |
+------------------------+--------------------------+



---

## 4. Now save it

The pipeline fits and transforms. It does not save.

In [9]:
import shutil

path = os.path.join(SCRATCH, "custom_pipeline")
shutil.rmtree(path, ignore_errors=True)

try:
    pipeline.save(path)
    print("saved")
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:220])

ValueError
('Pipeline write will fail on this pipeline because stage %s of type %s is not MLWritable', 'CustomTokenizer_a49ef6a6ec32', <class '__main__.CustomTokenizer'>)


`CustomTokenizer` is not `MLWritable`, so the pipeline that contains it is not writable either —
and neither is the fitted `PipelineModel`. A workflow that cannot be written cannot be moved to
another process, which is most of what the chapter says persistence is for.

Three things are missing from the four-line class, and they are separable:

1. **`inputCol` and `outputCol` are ordinary Python attributes**, not declared `Param` objects.
   The class therefore does not participate in MLlib's parameter system: there is nothing for
   `extractParamMap` to read, no documentation string, and nothing for a `ParamGridBuilder` to
   put in a grid.
2. **The standard column parameters are re-declared rather than inherited.** The mixins
   `HasInputCol` and `HasOutputCol` already define them, with the same names and semantics every
   other MLlib stage uses.
3. **The class inherits no persistence machinery.** `DefaultParamsReadable` and
   `DefaultParamsWritable` supply `save` and `load` for any stage whose whole state is its
   parameters.

The third is the one responsible for the failure just observed. The first two are what make the
third possible: `DefaultParamsWritable` writes *parameters*, so a class whose settings are not
parameters has nothing for it to write.

---

## 5. The same transformer, savable

Same four-line `_transform`. Everything added is bookkeeping.

In [10]:
from pyspark import keyword_only
from pyspark.ml.param.shared import HasInputCol, HasOutputCol
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable


class SavableTokenizer(Transformer, HasInputCol, HasOutputCol,
                       DefaultParamsReadable, DefaultParamsWritable):

    @keyword_only
    def __init__(self, inputCol=None, outputCol=None):
        super().__init__()
        self._set(**self._input_kwargs)

    def _transform(self, df):
        return df.withColumn(self.getOutputCol(),
                             F.split(F.lower(F.col(self.getInputCol())), " "))

In [11]:
# inputCol is now a real Param: it has a name, a documentation string and a current value,
# and it prints the way every built-in stage's parameters print.
tok = SavableTokenizer(inputCol="text", outputCol="words")
print(tok.explainParam("inputCol"))
print(tok.explainParam("outputCol"))

inputCol: input column name. (current: text)
outputCol: output column name. (default: SavableTokenizer_76db40e65392__output, current: words)


In [12]:
pipeline2 = Pipeline(stages=[
    SavableTokenizer(inputCol="text", outputCol="words"),
    StopWordsRemover(inputCol="words", outputCol="filtered"),
])
model2 = pipeline2.fit(df)

path2 = os.path.join(SCRATCH, "savable_pipeline")
shutil.rmtree(path2, ignore_errors=True)
model2.save(path2)
print("saved")

saved


In [13]:
from pyspark.ml import PipelineModel

reloaded = PipelineModel.load(path2)
print([type(s).__name__ for s in reloaded.stages])
reloaded.transform(df).select("text", "filtered").show(truncate=False)

['SavableTokenizer', 'StopWordsRemover']
+------------------------+--------------------------+
|text                    |filtered                  |
+------------------------+--------------------------+
|He went to the store.   |[went, store.]            |
|Dogs like to fight cats.|[dogs, like, fight, cats.]|
|When did you do that?   |[that?]                   |
+------------------------+--------------------------+



In [14]:
# Same output, before and after the round trip.
assert (model2.transform(df).select("filtered").collect()
        == reloaded.transform(df).select("filtered").collect())
print("the reloaded pipeline produces the same column")

the reloaded pipeline produces the same column


One caveat the cell above hides, because it ran in the kernel that defined the class. A saved
custom stage records its **fully qualified class name**, and loading it needs that class to be
importable. Here the name is `__main__.SavableTokenizer`, which resolves only inside this
notebook's own kernel. A custom stage meant to be reloaded elsewhere belongs in a module that
ships with the job, not in a notebook cell.

---

## 6. Where the code in `_transform` actually runs

Everything above used only Spark's own column functions, so the work happened inside the JVM.
The moment a custom stage calls out to an external library, two things change and neither is
visible in the source.

* The body of `_transform` executes **on the executors**, not on the driver. Every executor must
  therefore have the library installed. A missing import surfaces as a task failure at run time,
  a long way from the line that caused it.
* Anything loaded when the transformer is *constructed* becomes part of what is serialized and
  shipped with the closure. A linguistic model of a few hundred megabytes, loaded in `__init__`,
  is shipped **once per task** rather than once per executor. Nothing in the code says so, and
  the execution time says it very clearly.

This is why MLlib ships no lemmatizer and why its own documentation points at external libraries
for anything past tokenization and stop-word removal: the transformer interface makes wrapping
one trivial, and deploying one is a separate problem.

---

## What to take away

A transformer is `_transform`. Four lines is enough to fit and to apply, and it is not enough to
save: a stage becomes savable when its settings are declared `Param`s, inherited where MLlib
already defines them, and when it inherits `DefaultParamsReadable` and `DefaultParamsWritable`.

The hand-written assembler is the warning worth carrying forward. It appended the right numbers
with the wrong type and raised nothing.

In [15]:
spark.stop()